# Test de HybridIoTDataset

Teste le package `hybridiot_dataset` avec le fichier CSV de Toulouse Open Data.

In [1]:
# Uncomment if dependencies are not installed
# %pip install -r requirements.txt

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from hybridiot import MeteoDataset
from torch.utils.data import DataLoader
from datetime import datetime
import torch

## 1. Chargement du fichier CSV

In [3]:
CSV_PATH = '../data/toulouse_meteo_live_jan2023(5stations).csv'

ds = MeteoDataset.from_csv(CSV_PATH)
print(ds)

  Sampling frequency detected: 15 min
MeteoDataset(sensors=[Sensor(id='Toulouse Avenue De Grande Bretagne', virtual=False, pos=(43.6000, 1.4200, 139.35m)), Sensor(id='Toulouse Basso Cambo', virtual=False, pos=(43.5700, 1.3900, 157.2m)), Sensor(id='Toulouse Montaudran', virtual=False, pos=(43.5600, 1.4900, 143.55m)), Sensor(id='Toulouse Parc Jardin Des Plantes', virtual=False, pos=(43.5900, 1.4500, 143.47m)), Sensor(id='Toulouse Soupetard', virtual=False, pos=(43.6100, 1.4800, 142.61m))], values=tensor([[[1.3400e+01, 1.3200e+01, 1.2900e+01,        nan, 1.2700e+01],
         [1.3300e+01, 1.3300e+01, 1.2900e+01,        nan,        nan],
         [1.3300e+01, 1.3400e+01, 1.3000e+01,        nan, 1.2700e+01],
         ...,
         [8.0000e+00, 7.5000e+00, 8.1000e+00,        nan, 7.8000e+00],
         [7.9000e+00, 7.4000e+00, 8.0000e+00,        nan,        nan],
         [7.8000e+00, 7.4000e+00, 8.0000e+00,        nan, 7.8000e+00]],

        [[8.9000e+01, 8.9000e+01, 9.1000e+01,        nan, 

## 2. Description et qualité

In [4]:
print(ds.describe())

MeteoDataset 'toulouse_meteo_live_jan2023(5stations)'
  Source      : ../data/toulouse_meteo_live_jan2023(5stations).csv
  Sensors     : 5 stations
  Data types  : ['temperature', 'humidite', 'pression', 'pluie', 'pluie_intensite_max']
  Tensor      : (5, 2976, 5)  (n_types × n_timesteps × n_sensors)
  Timesteps   : 2976 (every 15 min)
  Time range  : 2023-01-01 00:00:00+00:00 -> 2023-01-31 23:45:00+00:00
  NaN rate    : 20.76%


In [5]:
print(ds.quality_report())


Quality report toulouse_meteo_live_jan2023(5stations)
  Sensors    : 5
  Data types : ['temperature', 'humidite', 'pression', 'pluie', 'pluie_intensite_max']
  Timesteps  : 2976 (every 15 min)
  Time range : 2023-01-01 00:00:00+00:00 -> 2023-01-31 23:45:00+00:00
  Tensor     : (5, 2976, 5)  (n_types, n_timesteps, n_sensors)
  NaN rate   : 20.76%
  ------------------------------------------------------------

  [Toulouse Avenue De Grande Bretagne]  (overall NaN rate: 0.0%)
    temperature               NaN:     1 (  0.0%)
    humidite                  NaN:     1 (  0.0%)
    pression                  NaN:     1 (  0.0%)
    pluie                     NaN:     1 (  0.0%)
    pluie_intensite_max       NaN:     1 (  0.0%)

  [Toulouse Basso Cambo]  (overall NaN rate: 0.0%)
    temperature               NaN:     0 (  0.0%)
    humidite                  NaN:     0 (  0.0%)
    pression                  NaN:     0 (  0.0%)
    pluie                     NaN:     0 (  0.0%)
    pluie_intensite_

## 3. Tensor Values

In [6]:
print(f'Shape: {tuple(ds.values.shape)}  → (n_types, n_timesteps, n_sensors)')
print(f'Data types: {ds.data_types}')
print(f'Sensors: {[s.id for s in ds.sensors]}')
print(f'NaN rate: {torch.isnan(ds.values).float().mean().item():.2%}')

Shape: (5, 2976, 5)  → (n_types, n_timesteps, n_sensors)
Data types: ['temperature', 'humidite', 'pression', 'pluie', 'pluie_intensite_max']
Sensors: ['Toulouse Avenue De Grande Bretagne', 'Toulouse Basso Cambo', 'Toulouse Montaudran', 'Toulouse Parc Jardin Des Plantes', 'Toulouse Soupetard']
NaN rate: 20.76%


## 4. Capteurs et StateHistory

In [7]:
for sensor in ds.sensors:
    print(sensor)

Sensor(id='Toulouse Avenue De Grande Bretagne', virtual=False, pos=(43.6000, 1.4200, 139.35m))
Sensor(id='Toulouse Basso Cambo', virtual=False, pos=(43.5700, 1.3900, 157.2m))
Sensor(id='Toulouse Montaudran', virtual=False, pos=(43.5600, 1.4900, 143.55m))
Sensor(id='Toulouse Parc Jardin Des Plantes', virtual=False, pos=(43.5900, 1.4500, 143.47m))
Sensor(id='Toulouse Soupetard', virtual=False, pos=(43.6100, 1.4800, 142.61m))


In [8]:
sensor = ds.sensors[0]
print(f'Capteur: {sensor.id}')
print(f'Position: lat={sensor.position.latitude}, lon={sensor.position.longitude}, alt={sensor.position.altitude}m')
print()

# StateHistory dérivé du tenseur pour une feature spécifique
print("StateHistory pour la température (10 premières valeurs):")
for ts, feat, is_valid in ds.get_state_history(sensor, feature='temperature')[:10]:
    print(f'  {ts}  {feat:<20}  {"OK" if is_valid else "NaN"}')
print()

# Fault rate per feature
print('Fault rate per feature:')
for feat in ds.data_types:
    rate = ds.fault_rate_for_sensor(sensor, feature=feat)
    print(f'  {feat:<25} {rate:.1%}')
print(f'  {"overall":<25} {ds.fault_rate_for_sensor(sensor):.1%}')
print()

sM = ds.get_sensor('Toulouse Montaudran')
print('Fault rate per feature for Toulouse Montaudran:')
for feat in ds.data_types:
    rate = ds.fault_rate_for_sensor(sM, feature=feat)
    print(f'  {feat:<25} {rate:.1%}')
print(f'  {"overall":<25} {ds.fault_rate_for_sensor(sM):.1%}')

Capteur: Toulouse Avenue De Grande Bretagne
Position: lat=43.6, lon=1.42, alt=139.35m

StateHistory pour la température (10 premières valeurs):
  2023-01-01 00:00:00+00:00  temperature           OK
  2023-01-01 00:15:00+00:00  temperature           OK
  2023-01-01 00:30:00+00:00  temperature           OK
  2023-01-01 00:45:00+00:00  temperature           OK
  2023-01-01 01:00:00+00:00  temperature           OK
  2023-01-01 01:15:00+00:00  temperature           OK
  2023-01-01 01:30:00+00:00  temperature           OK
  2023-01-01 01:45:00+00:00  temperature           OK
  2023-01-01 02:00:00+00:00  temperature           OK
  2023-01-01 02:15:00+00:00  temperature           OK

Fault rate per feature:
  temperature               0.0%
  humidite                  0.0%
  pression                  0.0%
  pluie                     0.0%
  pluie_intensite_max       0.0%
  overall                   0.0%

Fault rate per feature for Toulouse Montaudran:
  temperature               0.2%
  humidite 

## 5. PyTorch __getitem__ et DataLoader

In [9]:
# window_size=96 = 24h en 15min (default)
print(f'window_size : {ds.window_size}')
print(f'n_timesteps : {ds.n_timesteps}')
print(f'len(ds)     : {len(ds)}  (= n_timesteps - window_size)')

window_size : 96
n_timesteps : 2976
len(ds)     : 2880  (= n_timesteps - window_size)


In [10]:
# Un seul échantillon (input, target)
x, y = ds[0]
print(f'input  shape: {x.shape} = (n_types, window_size, n_sensors)')
print(f'target shape: {y.shape} = (n_types, n_sensors)')
print(f'dtype: {x.dtype}')

print("Signification:")
print(f"  input  = mesures des timesteps 0 à {ds.window_size - 1}")
print(f"  target = mesure au timestep {ds.window_size} (à prédire)")

input  shape: torch.Size([5, 96, 5]) = (n_types, window_size, n_sensors)
target shape: torch.Size([5, 5]) = (n_types, n_sensors)
dtype: torch.float32
Signification:
  input  = mesures des timesteps 0 à 95
  target = mesure au timestep 96 (à prédire)


In [11]:
# Sliding window: chaque indice avance d'un timestep
x0, y0 = ds[0]
x1, y1 = ds[1]
print('ds[0]: input timestep 0..95 -> target timestep 96')
print('ds[1]: input timestep 1..96 -> target timestep 97')
print()
# Vérification: le target de ds[0] doit être égale
# au dernier élément de l'input de ds[1]
print(f'y0 == x1[:, -1, :] : {torch.allclose(y0, x1[:, -1, :])}')

ds[0]: input timestep 0..95 -> target timestep 96
ds[1]: input timestep 1..96 -> target timestep 97

y0 == x1[:, -1, :] : True


In [12]:
# DataLoader, batch d’échantillons (input, target)
loader       = DataLoader(ds, batch_size=32, shuffle=True)
batch_x, batch_y = next(iter(loader))
print(f'batch input  shape: {batch_x.shape} = (batch, n_types, window_size, n_sensors)')
print(f'batch target shape: {batch_y.shape} = (batch, n_types, n_sensors)')

batch input  shape: torch.Size([32, 5, 96, 5]) = (batch, n_types, window_size, n_sensors)
batch target shape: torch.Size([32, 5, 5]) = (batch, n_types, n_sensors)


In [13]:
# Exemple avec une window_size différente
# Recréer le dataset avec des fenêtres de 4 timesteps (1 heure)
ds_small = MeteoDataset.from_csv(CSV_PATH, window_size=4)
x, y = ds_small[0]
print(f'window_size=4, input shape: {x.shape}, target shape: {y.shape}')

  Sampling frequency detected: 15 min
window_size=4, input shape: torch.Size([5, 4, 5]), target shape: torch.Size([5, 5])


## 6. get_data_window

In [14]:
# Que la temperature, tout le dataset
w1 = ds.get_data_window(data_types='temperature')
print(w1)
print(f'Shape: {tuple(w1.valeurs.shape)}  → (n_timesteps, n_sensors)')
print(w1.valeurs)

DataWindow(type='temperature', shape=(1, 2976, 5), sensors=5, timesteps=2976, duration=44625min)
Shape: (1, 2976, 5)  → (n_timesteps, n_sensors)
tensor([[[13.4000, 13.2000, 12.9000,     nan, 12.7000],
         [13.3000, 13.3000, 12.9000,     nan,     nan],
         [13.3000, 13.4000, 13.0000,     nan, 12.7000],
         ...,
         [ 8.0000,  7.5000,  8.1000,     nan,  7.8000],
         [ 7.9000,  7.4000,  8.0000,     nan,     nan],
         [ 7.8000,  7.4000,  8.0000,     nan,  7.8000]]])


In [15]:
# Que janvier 2020, toutes les features, toutes les stations
w2 = ds.get_data_window(
    start_time=datetime(2020, 1, 1, tzinfo=ds.start_time.tzinfo),
    end_time=datetime(2020, 1, 31, 23, 45, tzinfo=ds.start_time.tzinfo),
)
print(w2)
print(f'Shape: {tuple(w2.valeurs.shape)}  → (n_types, n_timesteps, n_sensors)')

DataWindow(EMPTY: time range outside dataset)
Shape: (5, 0, 5)  → (n_types, n_timesteps, n_sensors)


In [16]:
# Temperature, du 1er au 7 janvier 2023, pour les 2 premiers capteurs
w3 = ds.get_data_window(
    data_types='temperature',
    start_time=datetime(2023, 1, 1, tzinfo=ds.start_time.tzinfo),
    end_time=datetime(2023, 1, 7, 23, 45, tzinfo=ds.start_time.tzinfo),
    sensor_ids=[ds.sensors[0].id, ds.sensors[1].id],
)
print(w3)
print(f'Capteurs: {w3.sensor_ids}')
print(f'Durée: {w3.duration_minutes:.0f} minutes')

DataWindow(type='temperature', shape=(1, 672, 2), sensors=2, timesteps=672, duration=10065min)
Capteurs: ['Toulouse Avenue De Grande Bretagne', 'Toulouse Basso Cambo']
Durée: 10065 minutes


In [17]:
# Coverage per sensor pour la température pour chaque capteur
w = ds.get_data_window(data_types='temperature')
for sensor_id, coverage in w.coverage_per_sensor().items():
    print(f"  {sensor_id:<35} {coverage:.1%}")

  Toulouse Avenue De Grande Bretagne  100.0%
  Toulouse Basso Cambo                100.0%
  Toulouse Montaudran                 99.8%
  Toulouse Parc Jardin Des Plantes    0.3%
  Toulouse Soupetard                  96.1%


In [18]:
# Exemple avec une feature très mal renseignée
# Jardin des plantes n'a pas de données pour l'intensité de pluie maximale
w = ds.get_data_window(data_types='pluie_intensite_max', sensor_ids=['Toulouse Parc Jardin Des Plantes'])
print(w)
print(f'NaN rate: {w.nan_rate:.1%}')          
print(f'is_all_nan: {w.is_all_nan}')          
print(f'Coverage: {w.coverage_per_sensor()}') 

DataWindow(type='pluie_intensite_max', shape=(1, 2976, 1), sensors=1, timesteps=2976, duration=44625min)
NaN rate: 100.0%
is_all_nan: True
Coverage: {'Toulouse Parc Jardin Des Plantes': 0.0}


In [19]:
# Coverage pour plusieurs features du même capteur
w = ds.get_data_window(data_types=['temperature', 'pluie_intensite_max'], sensor_ids=['Toulouse Montaudran'])
print(w)
print(f"Coverage for 'temperature': {w.coverage_per_sensor(data_type='temperature')}")                 
print(f"Coverage for 'pluie_intensite_max': {w.coverage_per_sensor(data_type='pluie_intensite_max')}") 

DataWindow(type='temperature,pluie_intensite_max', shape=(2, 2976, 1), sensors=1, timesteps=2976, duration=44625min)
Coverage for 'temperature': {'Toulouse Montaudran': 0.9976478219032288}
Coverage for 'pluie_intensite_max': {'Toulouse Montaudran': 1.0}
